In [ ]:
# @title Step 1: Setup and Data Loading

# Install essential libraries (run once)
!pip install pandas numpy matplotlib seaborn scikit-learn joblib flask-ngrok

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Mount Google Drive to save your model permanently
from google.colab import drive
drive.mount('/content/drive')

# Load the dataset
# Option 1: Upload file directly to Colab
from google.colab import files
print("Please upload your 'hypertension_dataset.csv' file:")
uploaded = files.upload()

# Load the data into a pandas DataFrame
import io
df = pd.read_csv(io.BytesIO(uploaded['/content/hypertension_dataset.csv']))
print(f"Dataset shape: {df.shape}")
df.head()


In [ ]:
from google.colab import drive
import pandas as pd
df = pd.read_csv('/content/hypertension_dataset.csv')
print(f"Dataset shape: {df.shape}")
df.head()



In [ ]:
# @title Step 2: Exploratory Data Analysis

# Basic info about the dataset
print("Dataset Info:")
df.info()

print("\nDescriptive Statistics:")
df.describe(include='all')

# Visualize the target variable
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Has_Hypertension')
plt.title('Distribution of Hypertension (Target Variable)')
plt.show()

# Correlation heatmap (for numerical features)
plt.figure(figsize=(10,8))
# Select only numerical columns for correlation
numerical_df = df.select_dtypes(include=[np.number])
sns.heatmap(numerical_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()


In [ ]:
# @title Step 3: Data Preparation

# Make a copy to avoid modifying original data
data = df.copy()

# Handle missing values if any (example: drop them)
data = data.dropna()

# --- Encoding Categorical Features ---
# Identify categorical columns (example)
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
# Remove the target variable if it's categorical (we'll encode it separately)
if 'Has_Hypertension' in categorical_cols:
    categorical_cols.remove('Has_Hypertension')

# Use pandas get_dummies for one-hot encoding
data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

# Encode the target variable (if it's object type like 'Yes'/'No')
label_encoder = LabelEncoder()
data['Has_Hypertension'] = label_encoder.fit_transform(data['Has_Hypertension'])

print("Data after encoding:")
data.head()

In [ ]:
# @title Step 4: Model Training

# Define features (X) and target (y)
X = data.drop('Has_Hypertension', axis=1)
y = data['Has_Hypertension']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling (important for Logistic Regression)
# Identify columns to scale (usually all numerical features)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Cell 4: Train Model and CREATE SCALER
# Prepare features and target
X = data.drop('Has_Hypertension', axis=1)
y = data['Has_Hypertension']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# CREATE THE SCALER (THIS IS WHAT YOU NEED!)
scaler = MinMaxScaler()  # 👈 This creates the scaler object
X_train_scaled = scaler.fit_transform(X_train)  # 👈 This trains the scaler on your data
X_test_scaled = scaler.transform(X_test)

# Train model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print("✅ Scaler CREATED and trained successfully!")
print(f"Type of scaler: {type(scaler)}")
print(f"Scaler has learned min values: {scaler.data_min_}")
print(f"Scaler has learned max values: {scaler.data_max_}")

In [ ]:
# Cell 5: SAVE SCALER TO GOOGLE DRIVE (FIXED)
import os

# Create folder in Google Drive
folder_path = '/content/drive/MyDrive/hypertension_project/'
os.makedirs(folder_path, exist_ok=True)  # This creates the folder if it doesn't exist

# Define paths - notice these are .pkl files, NOT datasets!
model_path = folder_path + 'hypertension_model.pkl'      # 👈 This will save the MODEL
scaler_path = folder_path + 'hypertension_scaler.pkl'    # 👈 This will save the SCALER

# Save both files
joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)

print(f"✅ MODEL saved to: {model_path}")
print(f"✅ SCALER saved to: {scaler_path}")

# Check if files exist
if os.path.exists(model_path):
    print(f"📁 Model file size: {os.path.getsize(model_path)/1024:.2f} KB")
if os.path.exists(scaler_path):
    print(f"📁 Scaler file size: {os.path.getsize(scaler_path)/1024:.2f} KB")

In [ ]:
# @title Step 5: Save Model and Scaler

# Define paths in your Google Drive
model_path = '/content/drive/MyDrive/hypertension_project/hypertension_model.pkl'
scaler_path = '/content/drive/MyDrive/hypertension_project/hypertension_scaler.pkl'

# Save
joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)
print(f"Model and Scaler saved to Google Drive.")

In [ ]:
# @title Step 6: Deploy Flask App with ngrok

from flask import Flask, request, jsonify
from flask_ngrok import run_with_ngrok
import joblib
import numpy as np

# Load the saved model and scaler
model = joblib.load('/content/drive/MyDrive/hypertension_model.pkl')
scaler = joblib.load('/content/drive/MyDrive/scaler.pkl')

# Initialize Flask app
app = Flask(__name__)
run_with_ngrok(app)  # Start ngrok when app is run

@app.route('/')
def home():
    return "<h1>Hypertension Prediction API</h1><p>Use the /predict endpoint with POST request.</p>"

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Get JSON data from the request
        data = request.get_json()

        # Convert data to numpy array (ensure feature order matches training)
        # This is a simplified example. You'll need to structure this based on your features.
        features = np.array([[
            data.get('Age'),
            data.get('BMI'),
            # ... include all other features in the correct order
        ]])

        # Scale the features
        features_scaled = scaler.transform(features)

        # Make prediction
        prediction = model.predict(features_scaled)[0]
        probability = model.predict_proba(features_scaled)[0].max()

        # Interpret prediction
        result = "Hypertensive" if prediction == 1 else "Normal"
        confidence = round(probability * 100, 2)

        return jsonify({
            'prediction': result,
            'confidence': confidence,
            'status': 'success'
        })

    except Exception as e:
        return jsonify({'error': str(e), 'status': 'error'}), 400

# Run the app
print("Starting Flask app with ngrok...")
app.run()

In [ ]:
# @title Step 6: Deploy Flask App with ngrok

from flask import Flask, request, jsonify
from flask_ngrok import run_with_ngrok
import joblib
import numpy as np

# Load the saved model and scaler
model = joblib.load('/content/drive/MyDrive/hypertension_model.pkl')
scaler = joblib.load('/content/drive/MyDrive/scaler.pkl')

# Initialize Flask app
app = Flask(__name__)
run_with_ngrok(app)  # Start ngrok when app is run

@app.route('/')
def home():
    return "<h1>Hypertension Prediction API</h1><p>Use the /predict endpoint with POST request.</p>"

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Get JSON data from the request
        data = request.get_json()

        # Convert data to numpy array (ensure feature order matches training)
        # This is a simplified example. You'll need to structure this based on your features.
        features = np.array([[
            data.get('Age'),
            data.get('BMI'),
            # ... include all other features in the correct order
        ]])

        # Scale the features
        features_scaled = scaler.transform(features)

        # Make prediction
        prediction = model.predict(features_scaled)[0]
        probability = model.predict_proba(features_scaled)[0].max()

        # Interpret prediction
        result = "Hypertensive" if prediction == 1 else "Normal"
        confidence = round(probability * 100, 2)

        return jsonify({
            'prediction': result,
            'confidence': confidence,
            'status': 'success'
        })

    except Exception as e:
        return jsonify({'error': str(e), 'status': 'error'}), 400

# Run the app
print("Starting Flask app with ngrok...")
app.run()

In [ ]:
!pip install flask-ngrok

from flask import Flask, request, jsonify
from flask_ngrok import run_with_ngrok
import joblib
import numpy as np

app = Flask(__name__)
run_with_ngrok(app)  # Start ngrok when app is run

# Your routes here

if __name__ == '__main__':
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
Exception in thread Thread-3:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
            